Self-RAG is an advanced RAG technique where the LLM actively decides when to retrieve information and evaluates the retrieved information and its own generated answer before producing the final response.

Unlike basic RAG, where retrieval always happens, Self-RAG allows the model to decide whether retrieval is necessary and uses self-reflection/critique to improve the quality and factuality of the answer.


In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter

# SelfRAG only needs `retriever` and `llm`.
# Build them directly here instead of running the full pipeline notebooks.
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = TokenTextSplitter(chunk_size=256, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOllama(model="llama3:latest", temperature=0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7335.60it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from advanced_RAG.Self_RAG.Self_RAG import SelfRAG

print("Retriever:", type(retriever))
print("LLM:", type(llm))

Retriever: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
LLM: <class 'langchain_ollama.chat_models.ChatOllama'>


In [4]:
# Self-RAG ko test question
question = "What is task decomposition for LLM agents?"

# Existing retriever use garera documents retrieve gareko
retrieved_docs = retriever.invoke(question)

print(f"Retrieved documents: {len(retrieved_docs)}")

Retrieved documents: 3


In [5]:
# Retrieve bhayeko documents herna
for i, doc in enumerate(retrieved_docs, start=1):

    print(f"\n========== Document {i} ==========")
    print(doc.page_content[:500])


========== Document 1 ==========


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In

========== Document 2 ==========


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be f

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [7]:
# Question ko answer garna retrieval required cha ki chaina bhanera decide garne prompt

retrieval_decision_prompt = ChatPromptTemplate.from_template("""
You are a Self-RAG retrieval decision system.

Determine whether external documents are required
to answer the following question accurately.

Question:
{question}

Respond with only YES or NO.
""")

In [8]:
# Retrieval decision prompt lai existing LLM sanga connect gareko

retrieval_decision_chain = retrieval_decision_prompt | llm | StrOutputParser()

In [9]:
# Self-RAG ko retrieval decision test gareko

decision = retrieval_decision_chain.invoke({"question": question})

decision = decision.strip().upper()

print("Retrieval needed:", decision)

Retrieval needed: NO


In [10]:
# Retrieved document question sanga relevant cha ki chaina check garne prompt

relevance_prompt = ChatPromptTemplate.from_template("""
You are a document relevance evaluator.

Question:
{question}

Document:
{document}

Does this document contain information useful for answering
the question?

Respond with only YES or NO.
""")

In [11]:
# Relevance grader lai existing local LLM sanga connect gareko

relevance_chain = relevance_prompt | llm | StrOutputParser()

In [12]:
# Each retrieved document ko relevance test gareko

for i, doc in enumerate(retrieved_docs, start=1):

    grade = relevance_chain.invoke({"question": question, "document": doc.page_content})

    print(f"Document {i}: {grade.strip().upper()}")

Document 1: YES
Document 2: YES
Document 3: YES


In [13]:
# ANSWER GENERATION


# Retrieved documents lai context ma combine garera answer generate garne prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a helpful RAG assistant.

Answer the question using the provided context.

If the context does not contain enough information,
do not invent information.

Context:
{context}

Question:
{question}

Give a concise and accurate answer.
""")

# Generation prompt lai existing local LLM sanga connect gareko

generation_chain = generation_prompt | llm | StrOutputParser()

In [14]:
# Relevant documents ko content lai euta context ma combine gareko

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

# Context ra question LLM lai diyera answer generate gareko

answer = generation_chain.invoke({"context": context, "question": question})

print("Generated Answer:")
print(answer)

Generated Answer:
According to the provided context, task decomposition for LLM agents refers to the process of breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.


In [15]:
# ANSWER SUPPORT / FAITHFULNESS CHECK


# Generated answer retrieved context bata supported cha ki chaina check garne prompt

support_prompt = ChatPromptTemplate.from_template("""
You are a Self-RAG critic.

Determine whether the answer is fully supported by
the provided context.

Context:
{context}

Answer:
{answer}

Respond with only YES or NO.
""")

# Support checker lai existing LLM sanga connect gareko

support_chain = support_prompt | llm | StrOutputParser()

In [16]:
# Generated answer context bata supported cha ki chaina check gareko

support_result = support_chain.invoke({"context": context, "answer": answer})

support_result = support_result.strip().upper()

print("Answer supported:", support_result)

Answer supported: YES


In [17]:
# ==========================================
# ANSWER USEFULNESS CHECK
# ==========================================

# Answer le user ko question properly answer gareko cha ki chaina check garne prompt

usefulness_prompt = ChatPromptTemplate.from_template("""
You are a Self-RAG answer evaluator.

Determine whether the answer directly and adequately
answers the user's question.

Question:
{question}

Answer:
{answer}

Respond with only YES or NO.
""")

# Usefulness checker lai existing LLM sanga connect gareko

usefulness_chain = usefulness_prompt | llm | StrOutputParser()

In [18]:
# Generated answer useful cha ki chaina check gareko

usefulness_result = usefulness_chain.invoke({"question": question, "answer": answer})

usefulness_result = usefulness_result.strip().upper()

print("Answer useful:", usefulness_result)

Answer useful: YES


In [19]:
# SELF-CORRECTION


# Answer reliable chaina bhane question lai refine garera feri retrieval garne prompt

rewrite_prompt = ChatPromptTemplate.from_template("""
You are a query refinement system.

Rewrite the question so that it is more specific
and easier for a retrieval system to find relevant information.

Original question:
{question}

Return only the improved search query.
""")

# Query refinement prompt lai LLM sanga connect gareko

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

In [20]:
# Original question lai better retrieval ko lagi rewrite gareko

improved_question = rewrite_chain.invoke({"question": question})

improved_question = improved_question.strip()

print("Improved query:")
print(improved_question)

Improved query:
Here is the rewritten question:

"What is the concept of task decomposition in Large Language Model (LLM) architecture, specifically in the context of natural language processing and decision-making for artificial intelligence agents?"

This rewritten question is more specific and targeted, making it easier for a retrieval system to find relevant information.


In [21]:
# Improved query use garera feri documents retrieve gareko

retry_docs = retriever.invoke(improved_question)

print(f"Retrieved documents after retry: " f"{len(retry_docs)}")

Retrieved documents after retry: 3


In [22]:
# Retry retrieval bata aayeko documents herna

for i, doc in enumerate(retry_docs, start=1):

    print(f"\n========== Retry Document {i} ==========")
    print(doc.page_content[:500])


========== Retry Document 1 ==========
.
The design of generative agents combines LLM with memory, planning and reflection mechanisms to enable agents to behave conditioned on past experience, as well as to interact with other agents.

Memory stream: is a long-term memory module (external database) that records a comprehensive list of agents’ experience in natural language.

Each element is an observation, an event directly provided by the agent.
- Inter-agent communication can trigger new natural language statements.


Retrieval mod

========== Retry Document 2 ==========
.
The design of generative agents combines LLM with memory, planning and reflection mechanisms to enable agents to behave conditioned on past experience, as well as to interact with other agents.

Memory stream: is a long-term memory module (external database) that records a comprehensive list of agents’ experience in natural language.

Each element is an observation, an event directly provided by the agent.
- Inter